# 🏥 Logistic Regression — Diabetes Prediction

**Biomedical ML Toolkit** | C. Sathish Kumar

In this notebook we will:
1. Load the PIMA Diabetes Dataset
2. Preprocess the data
3. Train a Logistic Regression model
4. Evaluate with accuracy, confusion matrix, classification report
5. Predict on a new patient

**Open in Google Colab:** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

In [ ]:
# Install if running in Colab
# !pip install pandas scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print('Libraries loaded successfully!')

## Step 1: Load Dataset

In [ ]:
# Load from local (or from GitHub raw URL for Colab)
try:
    df = pd.read_csv('../datasets/diabetes.csv')
except:
    url = 'https://raw.githubusercontent.com/YOUR_USERNAME/biomedical-ml-toolkit/main/datasets/diabetes.csv'
    df = pd.read_csv(url)

print(f'Dataset shape: {df.shape}')
print(f'Diabetic patients: {df["Outcome"].sum()}')
print(f'Non-diabetic: {(df["Outcome"]==0).sum()}')
df.head()

## Step 2: Explore the Data

In [ ]:
# Plot distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']
for i, col in enumerate(cols):
    ax = axes[i//4][i%4]
    df.groupby('Outcome')[col].plot(kind='hist', ax=ax, alpha=0.6, bins=20, legend=True)
    ax.set_title(col)
    ax.legend(['Non-Diabetic', 'Diabetic'])
plt.tight_layout()
plt.suptitle('Feature Distributions by Diabetes Outcome', y=1.02, fontsize=14)
plt.show()

## Step 3: Preprocess Data

In [ ]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Replace medically impossible zero values
for col in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
    X[col] = X[col].replace(0, X[col].mean())

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Training: {len(X_train)} | Testing: {len(X_test)}')

## Step 4: Train Model

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)
print('Model trained!')

## Step 5: Evaluate Results

In [ ]:
y_pred = model.predict(X_test_s)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc*100:.2f}%')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Non-Diabetic','Diabetic']))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: No','Pred: Yes'],
            yticklabels=['Actual: No','Actual: Yes'])
plt.title(f'Confusion Matrix (Accuracy: {acc*100:.1f}%)', fontsize=13)
plt.tight_layout()
plt.show()

## Step 6: Feature Importance

In [ ]:
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_[0]}).sort_values('Coefficient')

plt.figure(figsize=(8,5))
colors = ['red' if c > 0 else 'steelblue' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Coefficients (Positive = Higher Diabetes Risk)', fontsize=12)
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()
print('Red bars = increases diabetes risk | Blue bars = decreases risk')

## Step 7: Predict New Patient

In [ ]:
# New patient: [Pregnancies, Glucose, BP, SkinThickness, Insulin, BMI, Pedigree, Age]
new_patient = pd.DataFrame([[2, 148, 72, 35, 94, 33.6, 0.627, 45]], columns=X.columns)
new_scaled  = scaler.transform(new_patient)
pred = model.predict(new_scaled)[0]
prob = model.predict_proba(new_scaled)[0][1]

print('=== NEW PATIENT PREDICTION ===')
print(f'Glucose: 148 mg/dL | BMI: 33.6 | Age: 45')
print(f'Prediction: {"DIABETIC ⚠️" if pred==1 else "NOT DIABETIC ✅"}')
print(f'Probability of Diabetes: {prob*100:.1f}%')